# In this notebook, All the Data preprocessing,cleaning stuffs are done !


In [ ]:
# we used used cars dataset 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder,StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
data=pd.read_csv('/Users/bibekacharya/Documents/Documents/Machine_Learning/Machine Learning projects/Linear Regression/vehicles.csv')
df=data.copy()
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
#lets drop some usless col that is not so imp for us 
df.drop(['url','region_url','image_url','lat','long'],axis=1,inplace=True)
df.head() 

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df['county'].isnull().sum()


In [ ]:
df.drop('county',axis=1,inplace=True)

In [ ]:
df.head(3)

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()


In [ ]:
# looking at null, VIN and size will also get dropped 
df.drop(['size','VIN'],axis=1,inplace= True)


In [ ]:
# We have abduance of categorical columns compared to other and they are are also filled with NULL relatively 

categorical_col=df.drop(['id','price','year','odometer'],axis=1).columns


In [ ]:

for category in categorical_col:
    df[category]=df[category].fillna('Unknown')
    

In [ ]:
categorical_col

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['year']=df['year'].fillna(df['year'].median())
df['odometer']=df['odometer'].fillna(df['odometer'].mean())

In [ ]:
df.head()

In [ ]:
# lets try to get some insights form our data 

# univariate analysis 

df['odometer'].describe()
df['odometer'].hist(bins=10)
plt.show()

Graph shows data is highly right skewed

In [ ]:
# lets see the price distribution as well from the data 

sns.histplot(df['price'],bins=10)

price is also highly right skewed, some hiking realistically high as well 

In [ ]:
df['price'].skew()

In [ ]:
df.head()

In [ ]:
df['type'].value_counts()

### Univatiate Analysis


In [ ]:
# univariate categorical Analysis 

df['type'].value_counts().plot(kind='bar')
plt.title('Types of vehicle present in data ')
plt.ylabel('values count')

In [ ]:
df['manufacturer'].value_counts()

In [ ]:
df['manufacturer'].value_counts().head(10).sort_values().plot(kind='barh',color='red')
plt.title("The top 10 vehicle manufacturers")
plt.ylabel('Vehicle count ')

In [ ]:
sns.boxplot(df['price'],orient='h')

In [ ]:
df['price'].value_counts().sort_values(ascending=False)

In [ ]:
# we have extreme outliers so lets handle them 

df = df[(df['price'] > 500) & (df['price'] < 100000)]

In [ ]:
sns.boxplot(df['price'],orient='h') #boxplot after we removed the outlier d

There are some extreme outliers present in price of the vehicles the data 

### Bivariate Analysis

In [ ]:
df.tail()

In [ ]:
# how's the correlatin between year and price 
df[['year','price']].corr()

In [ ]:
# negative correlation so we can see year and price dosent have a strong correlation with each other.

In [ ]:
# Fuel type vs price 
df.groupby('fuel')['price'].median().sort_values().plot(kind='bar')
plt.title("Fuel vs price: price of car based on fuel type ")
plt.ylabel('price of cars ')
plt.tight_layout()
plt.show()

Disel car tend to have more price. Petrol and hybrid vehicels price are in similar range 


In [ ]:
# does condition effect price of the car ?? 
df.groupby('condition')['price'].median()

In [ ]:


df.groupby('condition')['price'].median().plot(kind='line')
plt.ylabel('median prices of cars')
plt.title("Does condition of car affect price ?")


Yes! definitely good condition cars has the higest peak for price. Salvage car(damaged) has a significant downfall in the price . 

In [ ]:
# Does having more odometer number reading means the car is significantly older ? 
# we assumed the year means the model year of the car that was launched 

In [ ]:
# Does having more odometer number reading means the car is significantly older ? dd
df.groupby('year')['odometer'].mean().plot()

In [ ]:
df['transmission'].value_counts()

In [ ]:
# which state has higher price listings ?
df.groupby('state')['price'].mean().sort_values(ascending=False).head(10).plot(kind='barh')
plt.title('Top 10 States with Highest Mean Car Prices')
plt.xlabel('Mean Price')
plt.show()

In [ ]:
pivot = df.pivot_table(
    values='price',
    index='fuel',
    columns='transmission',
    aggfunc='mean'
)

sns.heatmap(pivot, annot=True, fmt='.0f')
plt.title('Mean Vehicle Price Across Fuel Types and Transmission Types')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate Analysis 

sns.heatmap(df[['price','year','odometer','id']].corr(), annot=True)

In [ ]:
for col in categorical_col:
    print(col, df[col].nunique()) #check unique values in colummn 



In [ ]:
# now we will proceed toward encoding of data. Here we will drop the feature which are not useful for our model 

df.drop(['description','posting_date','model','region'],axis=1,inplace=True)


''' why we dropped them ?? 

for trainig model, model needs numbers not categorical values and those dropped col had like hundreds of categorical value 
and also they were not so useful too .. fo those thousands of categoical value during encoding eithr we use 
ordinal or nomimal based on data. 

either we assign 0,1,2 new numbers for new val or construct n-1 col for N feature . so description had like 3245k 
unique values sp it meant around 320k col so chences our model would explode.  '''

In [ ]:
df.drop(['id','paint_color'],axis=1 ,inplace=True ) # id and paint color has no such importance 

In [ ]:
df.head()

In [ ]:
X=df.drop('price',axis=1)#feature col
y=df['price']# target col 

In [ ]:
# train test split 
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=43)

In [ ]:
# to check if ordinal or nomial values 
print(df['drive'].unique())
print(df['title_status'].unique())
print(df['transmission'].unique())
print(df['condition'].unique())
print(df['fuel'].unique())
print(df['type'].unique())
print(df['cylinders'].unique())

In [ ]:
# Encode the data 
# ordinal- Has realtion or order exists and Nomial- no realtion or order exits

from sklearn.compose import ColumnTransformer

# categories for ordinal so that it becomes neat X = df.drop('price', axis=1)


cylinders_order = ['Unknown', '3 cylinders', '4 cylinders', '5 cylinders',
                   '6 cylinders', '8 cylinders', '10 cylinders', '12 cylinders', 'other']
title_status_order = ['parts only', 'missing', 'salvage', 'lien', 'rebuilt', 'clean', 'Unknown']
condition_order = ['salvage', 'fair', 'good', 'excellent', 'like new', 'new', 'Unknown']

ct = ColumnTransformer(transformers=[
    ('tnf1', OrdinalEncoder(categories=[cylinders_order, title_status_order, condition_order]),
                            ['cylinders', 'title_status', 'condition']),
    ('tnf2', OneHotEncoder(drop='first', handle_unknown='ignore',sparse_output= False),
                           ['manufacturer', 'fuel', 'transmission', 'drive', 'type', 'state']),
    ('tnf3', StandardScaler(), ['year', 'odometer'])
], remainder='drop')

X_train = ct.fit_transform(X_train)  # fit AND transform on train
X_test = ct.transform(X_test)        # only transform on test (never fit!)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
import pickle

with open('used_cars.ipynb', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'ct': ct  
    }, f)